In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime
import pickle, os, json

# ML
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.dummy import DummyClassifier
import xgboost as xgb
import lightgbm as lgb

# MLflow
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm
from hmmlearn.hmm import GaussianHMM
import numpy as np

# ── CONFIG ──────────────────────────────────────────────────────────────
RANDOM_SEED       = 42
WFV_START_YEAR    = 2015        # First test year (need ~15 years of training data)
WFV_END_YEAR      = 2024        # Last test year
MIN_TRAIN_YEARS   = 5           # Minimum years required before first test fold
MLFLOW_EXP_NAME   = 'market_regime_prediction'

REGIME_NAMES      = {0: 'Bear', 1: 'Sideways', 2: 'Bull'}
REGIME_COLOR      = {'Bear': '#e74c3c', 'Sideways': '#f39c12', 'Bull': '#2ecc71'}

os.makedirs('data',   exist_ok=True)
os.makedirs('plots',  exist_ok=True)
os.makedirs('models', exist_ok=True)

print('Config OK')
print(f'MLflow version: {mlflow.__version__}')

Config OK
MLflow version: 3.10.0


In [2]:
# Load feature matrix from Notebook 02
df = pd.read_csv('data/02_feature_matrix.csv', index_col=0, parse_dates=True)

# Load selected feature list (pruned correlated features)
with open('data/selected_features.pkl', 'rb') as f:
    selected_features = pickle.load(f)

# Validate columns exist
missing = [c for c in selected_features if c not in df.columns]
if missing:
    print(f'WARNING: {len(missing)} features missing from matrix: {missing[:5]}')
    selected_features = [c for c in selected_features if c in df.columns]

TARGET_COL   = 'target_regime_code'
TARGET_NAME  = 'target_regime_name'

print(f'Dataset shape    : {df.shape}')
print(f'Features used    : {len(selected_features)}')
print(f'Date range       : {df.index[0].date()} → {df.index[-1].date()}')
print(f'Target classes   : {df[TARGET_COL].value_counts().to_dict()}')
df.tail(3)

Dataset shape    : (4119, 78)
Features used    : 52
Date range       : 2008-03-04 → 2024-12-27
Target classes   : {2.0: 1727, 0.0: 1697, 1.0: 695}


,Close,target_regime_code,target_regime_name,ret_1d,ret_2d,ret_3d,ret_5d,ret_10d,ret_21d,ret_63d,...,regime_lag2,regime_lag3,regime_lag5,regime_duration,was_bull_lag1,was_sideways_lag1,was_bear_lag1,was_bull_lag2,was_sideways_lag2,was_bear_lag2
Date,,,,,,,,,,,,,,,,,,,,,
2024-12-24,23727.650391,2.0,Bull,0.007011,-0.008312,-0.018577,-0.037789,-0.035791,-0.006454,-0.082296,...,0.0,0.0,0.0,4.0,1.0,0.0,0.0,0.0,0.0,1.0
2024-12-26,23750.199219,2.0,Bull,-0.001087,0.005924,-0.009398,-0.025316,-0.036514,-0.020616,-0.089109,...,2.0,0.0,0.0,4.0,1.0,0.0,0.0,1.0,0.0,0.0
2024-12-27,23813.400391,2.0,Bull,0.000950,-0.000137,0.006874,-0.018714,-0.036853,-0.018534,-0.088211,...,2.0,2.0,0.0,4.0,1.0,0.0,0.0,1.0,0.0,0.0


In [3]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df[selected_features], df[TARGET_COL],test_size=0.2, random_state=RANDOM_SEED, stratify=df[TARGET_COL])

In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.values)
X_test_scaled = scaler.transform(X_test.values)

In [9]:
from hmmlearn.hmm import GaussianHMM
import numpy as np

model = GaussianHMM(n_components=3, covariance_type="full", n_iter=1000)
model.fit(X_train_scaled)

hidden_states = model.predict(X_train_scaled)

In [10]:
print("Hidden states:", hidden_states)

Hidden states: [2 0 0 ... 0 0 0]


In [11]:
transition_matrix = model.transmat_
print(transition_matrix)

[[0.77137489 0.09293939 0.13568573]
 [0.80537371 0.0838842  0.11074209]
 [0.77130784 0.0807101  0.14798206]]


In [12]:
def predict_next_regime(model, X):
    logprob, state_probs = model.score_samples(X)
    current_prob = state_probs[-1]
    next_prob = np.dot(current_prob, model.transmat_)
    return next_prob
next_regime_prob = predict_next_regime(model, X_train_scaled)
print("Next regime probabilities:", next_regime_prob)

Next regime probabilities: [0.77137489 0.09293939 0.13568573]


In [14]:
predicted_state = np.argmax(next_regime_prob)
print("Predicted next regime state:", predicted_state)

Predicted next regime state: 0
